# Contrastive Mask Visualization

Runs the full CDEA pipeline on a small batch and shows interactive Plotly figures:
1. **Sample gallery** — input image, per-class evidence heatmaps, shared/unique overlays
2. **Grad-CAM vs IG side-by-side** — same image, both evidence providers
3. **Probability split bar chart** — shared-only vs shared+unique per class
4. **Pairwise margin heatmap** — 'why class k rather than l?' for all pairs

**Color code for overlays:** green = unique evidence (discriminates this class),  
red = shared evidence (common to multiple top classes), yellow = both.

In [ ]:
# --- Configuration ---
DATASET       = 'cifar10'       # mnist | cifar10 | pets | stanford_dogs
MODEL_NAME    = 'resnet18'
PRETRAINED    = True            # False = random init (fast smoke test)
BATCH_SIZE    = 8               # number of images to explain
NUM_STEPS     = 25              # allocator optimization steps
MAX_CLASSES   = 3               # top-k classes to show per image
GAME_MODE     = 'mixed'         # cooperative | mixed | competitive
EVIDENCE_MODE = 'gradcam'       # gradcam | ig  (used for gallery + prob split)
IG_STEPS      = 24

In [ ]:
from pathlib import Path
import sys
import torch
import torch.nn.functional as F

REPO = Path('/Users/ssuresh/gambit')
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from core.runner import CDEAExplainer
from core.hypotheses import TopMSelector
from core.device import get_device
from core.game_modes import resolve_contrastive_game
from core.interaction import get_interaction
from core.visualization import (
    show_explanation_gallery,
    show_gradcam_vs_ig,
    show_probability_split,
    show_pairwise_margin,
)
from modality.grid_regions import VisionGridUnitSpace
from instantiations.contrastive.objective import ContrastiveObjective
from instantiations.contrastive.allocator import OptimizationAllocator
from base_evidence.gradcam_regions import GradCAMRegionsProvider
from base_evidence.integrated_gradients_regions import IntegratedGradientsRegionsProvider
from examples.contrastive_explanation import (
    TV_INPUT_SIZE, _load_batch, get_torchvision_model,
)

device = get_device()
print('Device:', device)

In [ ]:
# --- Load data and model ---
x, class_names, num_classes = _load_batch(DATASET, BATCH_SIZE, device)
if x.shape[-1] != TV_INPUT_SIZE:
    x = F.interpolate(x, size=(TV_INPUT_SIZE, TV_INPUT_SIZE),
                      mode='bilinear', align_corners=False)

model = get_torchvision_model(MODEL_NAME, num_classes, pretrained=PRETRAINED).to(device).eval()
print(f'Model: {MODEL_NAME} ({"pretrained" if PRETRAINED else "random init"})')
print(f'Dataset: {DATASET} | Classes: {num_classes} | Batch: {x.shape}')

with torch.no_grad():
    logits_all = model(x)
    probs_all  = torch.softmax(logits_all, dim=-1)
    pred_ids   = logits_all.argmax(dim=-1).cpu()
print('Predicted classes:', [class_names[i] for i in pred_ids.tolist()])

In [ ]:
# --- Build explainer ---
GRID_H, GRID_W = 7, 7
game_cfg = resolve_contrastive_game(GAME_MODE)

unit_space = VisionGridUnitSpace(GRID_H, GRID_W, baseline='blur')
selector   = TopMSelector(m=min(MAX_CLASSES, num_classes))

if EVIDENCE_MODE == 'gradcam':
    provider = GradCAMRegionsProvider(GRID_H, GRID_W)
else:
    provider = IntegratedGradientsRegionsProvider(GRID_H, GRID_W, steps=IG_STEPS, baseline='zero')

objective = ContrastiveObjective(
    lambda_suff=1.0, lambda_margin=1.0,
    lambda_sparse=0.05, lambda_overlap=0.2,
    lambda_mass=game_cfg.lambda_mass if hasattr(game_cfg, 'lambda_mass') else 2.0,
)
allocator = OptimizationAllocator(
    objective=objective, num_steps=NUM_STEPS, lr=0.3,
    use_shared=game_cfg.use_shared,
    lambda_disjoint=game_cfg.lambda_disjoint,
)
explainer = CDEAExplainer(
    model=model, unit_space=unit_space, selector=selector,
    base_evidence=provider, allocator=allocator, objective=objective,
    interaction=get_interaction('none'),
    normalize_evidence=True, device=device,
)

explanation = explainer.explain(x)
print('Explanation complete.')
print({k: float(v.mean()) for k, v in explanation.metrics.items()
       if isinstance(v, torch.Tensor) and v.numel() == 1})

## 1. Sample Gallery — Evidence Heatmaps + Shared/Unique Overlays

Each row = one image from the batch.  
**Column 1:** Input image with predicted class.  
**Columns 2–N:** Evidence heatmap (hot colormap) for each top class.  
**Last columns:** Overlay — green=unique, red=shared, yellow=both.  
Hover over any cell to see class label and probability.

In [ ]:
out_dir = REPO / 'examples' / 'out'
out_dir.mkdir(parents=True, exist_ok=True)

fig = show_explanation_gallery(
    x, explanation, unit_space,
    class_names=class_names,
    probs=probs_all.detach().cpu(),
    n_samples=min(4, BATCH_SIZE),
    max_classes=MAX_CLASSES,
    title=f'CDEA Gallery — {DATASET} / {EVIDENCE_MODE} / {GAME_MODE}',
    out_path=out_dir / f'gallery_{DATASET}_{EVIDENCE_MODE}.html',
)
fig.show()

## 2. Grad-CAM vs Integrated Gradients — Side by Side

Runs both evidence providers on the same image and compares:
- Raw evidence heatmap from each provider
- The resulting unique/shared overlay from each provider

Useful for understanding how the choice of base evidence affects the allocation.

In [ ]:
SAMPLE_IDX = 0  # which image from the batch to compare

providers = {
    'Grad-CAM': GradCAMRegionsProvider(GRID_H, GRID_W),
    f'IG ({IG_STEPS} steps)': IntegratedGradientsRegionsProvider(
        GRID_H, GRID_W, steps=IG_STEPS, baseline='zero'),
}

explanations_by_provider = {}
for name, prov in providers.items():
    expl_i = CDEAExplainer(
        model=model, unit_space=unit_space, selector=selector,
        base_evidence=prov, allocator=allocator, objective=objective,
        interaction=get_interaction('none'),
        normalize_evidence=True, device=device,
    )
    explanations_by_provider[name] = expl_i.explain(x)
    print(f'{name} done.')

fig = show_gradcam_vs_ig(
    x, explanations_by_provider, unit_space,
    class_names=class_names,
    sample_idx=SAMPLE_IDX,
    max_classes=MAX_CLASSES,
    title=f'Grad-CAM vs IG — Sample {SAMPLE_IDX} — {DATASET}',
    out_path=out_dir / f'gradcam_vs_ig_{DATASET}.html',
)
fig.show()

## 3. Probability Split — Shared-Only vs Shared+Unique

For each top class, shows how much of the model's confidence comes from:
- **Shared evidence alone** — regions common to multiple hypotheses
- **Shared + unique evidence** — adding the class-specific regions

The delta shows the discriminative contribution of unique evidence.

In [ ]:
fig = show_probability_split(
    explanation,
    class_names=class_names,
    sample_idx=0,
    title=f'Probability Split — {DATASET}',
    out_path=out_dir / f'prob_split_{DATASET}.html',
)
if fig is not None:
    fig.show()

## 4. Pairwise Margin Heatmap — 'Why k Rather Than l?'

Each cell [k, l] shows the logit margin of class k over class l  
when the model sees only `keep(m_shared + m_unique[k])`.  

**Green (positive)** = class k wins over l under its own mask.  
**Red (negative)** = the mask fails to discriminate k from l — a diagnostic for weak explanations.

In [ ]:
fig = show_pairwise_margin(
    explanation,
    class_names=class_names,
    sample_idx=0,
    title=f'Pairwise Margin — {DATASET}',
    out_path=out_dir / f'pairwise_margin_{DATASET}.html',
)
if fig is not None:
    fig.show()